# LexiAssist: handwriting screening indicator for possible dyslexia-related patterns

End-to-end notebook: data → splits → models → evaluation → XAI → export → Streamlit app files.

**Important:** this is an educational screening aid, **not a diagnosis**. The labels come from small datasets (assessor-labelled or collected at different sites), so results must be reported with care.

**Before you run:** switch to a GPU runtime (*Runtime → Change runtime type → T4 GPU*), and download the datasets into your Drive (see Phase 2).

## Phase 1: Setup

### Theory: The toolchain

**1. What this code does**

Installs the libraries the notebook needs beyond what Google Colab ships with: `grad-cam` and `captum` for explainability, `scikit-learn` for splits and metrics, `pandas` for tables, `matplotlib` for plots, and OpenCV (Open Source Computer Vision Library) for image operations used by `grad-cam`. PyTorch and torchvision are already installed on Colab.

**2. Why it's needed**

Without these packages the later `import` lines fail. More subtly, *which version* you get matters: an unpinned `pip install` gives you whatever is newest today, so the same notebook can behave differently next year.

**3. The theory**

This notebook is a standard deep learning stack:

| Layer | Library | Role |
|---|---|---|
| Tensors + automatic differentiation | `torch` | Stores data as multi-dimensional arrays (tensors) and records every operation so it can compute gradients automatically (*autograd*) |
| Vision models + transforms | `torchvision` | Pretrained architectures (ResNet18) and image transforms (resize, normalise) |
| Classical machine learning (ML) utilities | `scikit-learn` | Cross-validation splitters and metrics such as ROC AUC |
| Explainable AI (XAI) | `grad-cam`, `captum` | Heatmaps showing which pixels influenced a prediction |

The key idea behind PyTorch is **reverse-mode automatic differentiation**: during the forward pass it builds a graph of operations; calling `loss.backward()` walks that graph backwards applying the chain rule, giving $\partial \text{loss} / \partial \theta$ for every parameter $\theta$. Everything in Phase 5 depends on this.

**4. How it maps to the code**

- `!pip -q install ...`: the `!` runs a shell command from the notebook; `-q` keeps the output quiet.
- `grad-cam` is the PyPI (Python Package Index) name; you import it as `pytorch_grad_cam` (see `app/explain.py`).
- `captum` is only used in Phase 9 (`IntegratedGradients`).

**5. Common pitfalls**

- **Package name vs import name.** `pip install grad-cam` but `import pytorch_grad_cam`. Installing a non-existent name such as `pytorch-grad-cam` fails.
- **Unpinned versions.** For a write-up you want reproducibility: record versions with `pip freeze > outputs/requirements-lock.txt` after a successful run.
- **Restart after installing.** If a package was already imported in the session, a newly installed version is not picked up until the runtime restarts.

In [ ]:
!pip -q install grad-cam captum scikit-learn pandas matplotlib opencv-python-headless

### Theory: Reproducibility, devices and hyperparameters

**1. What this code does**

Imports everything, decides where files live (Google Drive on Colab, the project folder locally), creates the data/checkpoint/output/app folders, and defines `CFG`, the dictionary of *hyperparameters* (settings you choose, as opposed to *parameters* the model learns). It also seeds every random number generator and picks the device: a GPU (graphics processing unit) via CUDA (NVIDIA's GPU computing platform) if available, otherwise the CPU (central processing unit).

**2. Why it's needed**

Deep learning is full of randomness: weight initialisation, data shuffling, augmentation, dropout. Without seeds, two runs give different numbers and you cannot tell whether a change helped or you just got lucky. On a dataset of a few hundred images that luck can be several percentage points.

**3. The theory**

*Hyperparameters in `CFG`:*

| Name | Meaning | Intuition |
|---|---|---|
| `img_size=224` | Images are resized to 224×224 pixels | The size ResNet was pretrained on |
| `batch_size=16` | Images per gradient step (a *mini-batch*) | Smaller batch = noisier but more frequent updates |
| `n_folds=5` | Number of cross-validation (CV) folds | Each fold is the test set once |
| `patience=7` | Epochs without improvement before stopping | Early stopping tolerance |
| `stages_*` | `(epochs, learning_rate, freeze_backbone)` per stage | Two-stage transfer learning for ResNet18 |

An **epoch** is one full pass over the training set. With $N$ training images and batch size $B$ there are $\lceil N/B \rceil$ gradient steps per epoch.

*Mini-batch gradient descent.* The true gradient is an average over the whole dataset; a mini-batch gives an unbiased but noisy estimate:

$$\hat g = \frac{1}{B}\sum_{i \in \text{batch}} \nabla_\theta \ell(x_i, y_i; \theta), \qquad \operatorname{Var}[\hat g] \propto \frac{1}{B}$$

Some noise is helpful (it can push the optimiser out of sharp minima), too much makes training unstable.

*Seeding.* A pseudo-random generator produces the same sequence from the same seed. `set_seed` seeds Python's `random`, NumPy and PyTorch (CPU and all GPUs). Some GPU kernels (from the cuDNN library, NVIDIA's deep-neural-network library) are still non-deterministic, so expect tiny run-to-run differences even with seeds.

**4. How it maps to the code**

- The `try: from google.colab import drive ... except ImportError` block chooses `ROOT`.
- `for d in (DATA_DIR, CKPT_DIR, OUT_DIR, APP_DIR): d.mkdir(...)` creates folders.
- `set_seed(s)` seeds `random`, `np.random`, `torch.manual_seed`, `torch.cuda.manual_seed_all`.
- `DEVICE` is used everywhere with `.to(DEVICE)`.

**5. Common pitfalls**

- **Reporting one seed.** With small data, results from a single seed can mislead. The fold-by-fold standard deviation in Phase 5 is a first look at this variability.
- **Full determinism costs speed.** `torch.backends.cudnn.deterministic = True` makes runs repeatable but slower; it is not set here.
- **Running on CPU by accident.** Training ResNet18 on CPU for 5 folds × 2 models is very slow. Check the printed `Device:` line.

In [ ]:
import os, re, sys, copy, random, json, shutil
from pathlib import Path

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,
                             precision_score, f1_score, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/LexiAssist")
except ImportError:
    ROOT = Path.cwd()   # local run: start Jupyter from the LexiAssist folder
DATA_DIR  = ROOT / "data"
CKPT_DIR  = ROOT / "checkpoints"
OUT_DIR   = ROOT / "outputs"
APP_DIR   = ROOT / "app"
for d in (DATA_DIR, CKPT_DIR, OUT_DIR, APP_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    seed=42, img_size=224, batch_size=16, n_folds=5,
    patience=7, num_workers=2,
    # (epochs, learning_rate, freeze_backbone)
    stages_resnet=[(8, 1e-3, True), (25, 1e-4, False)],
    stages_cnn=[(40, 1e-3, False)],
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
print("Device:", DEVICE)

## Phase 2: Data

**Download these manually and put them in Drive under `LexiAssist/data/`:**

1. **Main set:** *Potential Dysgraphia Handwriting Dataset of School-Age Children* (Mendeley Data, DOI 10.17632/39hr8dx76p.1). Unzip into `data/dysgraphia/`.
2. **External test set:** *Multilingual AI-Based Dyslexia Detection* (Mendeley Data, DOI 10.17632/b37s8b94nv.1). Unzip into `data/bilingual/`.
3. **Optional:** the Kaggle *Dyslexia Handwriting Dataset* (letter-level) into `data/kaggle_letters/`.

The code expects one sub-folder per class (e.g. `potential dysgraphia/`, `low potential dysgraphia/`). If your folder names differ, edit `LABEL_MAP` below. Label `1` = positive class (possible indicators), `0` = negative class.

### Theory: Labels, metadata and data leakage

**1. What this code does**

Walks the dataset folder, keeps image files, and turns each one into a row of a table (`df`) with its path, label (`1` = possible indicators, `0` = low indicators), a writer ID and the source dataset. The label comes from the folder name via `LABEL_MAP`; the writer ID comes from the filename via `WRITER_REGEX`.

**2. Why it's needed**

Models learn from `(image, label)` pairs, so the labels must be right. The writer ID is just as important: it is what later stops the same child's handwriting from appearing in both training and test data. Get it wrong and every metric in this notebook is inflated.

**3. The theory**

*Supervised learning.* We want a function $f_\theta$ that maps an image $x$ to a label $y$. We fit $\theta$ on examples and hope it **generalises** to new children. The standard assumption is that examples are IID (independent and identically distributed): each one is a fresh, independent draw from the same population.

*Why handwriting breaks IID.* Several samples from one child share that child's pen grip, letter shapes, slant, pressure and paper. They are *correlated*, not independent. The real unit of generalisation is the **writer**, not the image.

*Data leakage* is when information about the test set reaches training. Here, if child A's sentence 1 is in training and sentence 2 is in test, the model can score well by recognising child A's *style* rather than learning anything about indicators.

```
Leaky split (by image)              Writer-aware split (by child)
train: A1 A2 B1 C1 C2               train: A1 A2 A3 B1 B2
test : A3 B2 C3   <- A, B, C seen   test : C1 C2 C3   <- C never seen
```

*Labels are not ground truth.* The main dataset's labels are **assessor judgements of potential dysgraphia** (a writing difficulty), not clinical dyslexia diagnoses. Treat them as noisy: some "0" children may have difficulties and some "1" children may not. *Label noise* puts a ceiling on achievable accuracy and makes small differences between models unreliable.

**4. How it maps to the code**

- `norm(s)` lowercases folder names and turns `_`/`-` into spaces so `Low_Potential-Dysgraphia` matches `"low potential dysgraphia"`.
- `LABEL_MAP` maps normalised folder names to `0`/`1`; folders not listed are skipped and printed.
- `writer_id(p)` applies `WRITER_REGEX` to the filename. **With `WRITER_REGEX = None` it falls back to `p.stem`, so every image becomes its own "writer".**
- The `WARNING: no writer IDs detected` message fires exactly in that case.

**5. Common pitfalls**

- **Leaving `WRITER_REGEX = None`.** This is the single biggest risk in this notebook: the writer-aware split silently becomes an image-level split. Inspect the printed filenames and set the regex if they encode a child ID. If they don't, say so in your write-up and treat all metrics as optimistic upper bounds.
- **Wrongly inferred IDs.** A regex that groups two different children under one ID (or splits one child into two IDs) introduces its own leakage or noise. Spot-check a few groups by eye.
- **Folder name mismatches.** A typo in `LABEL_MAP` silently drops a whole class. Check the printed class counts.
- **Children's data.** These are images of minors' work. Use them only under the dataset licence, keep them out of public repositories, and never try to re-identify a child.

*Deeper dive:* `theory/05_validation_and_leakage.md`.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def norm(s):
    return re.sub(r"[_\-]+", " ", s.lower()).strip()

# keys are normalised folder names (lowercase, "_" and "-" → space)
LABEL_MAP = {
    "potential dysgraphia": 1, "low potential dysgraphia": 0,
    "yes": 1, "no": 0,
    "dyslexic": 1, "non dyslexic": 0,
}

# If filenames contain a writer/child ID, put a regex with ONE capture group here,
# e.g. r"^(\d+)_"  or  r"child(\d+)". Look at the printed filenames first.
WRITER_REGEX = None

def writer_id(p: Path):
    if WRITER_REGEX:
        m = re.search(WRITER_REGEX, p.stem)
        if m:
            return m.group(1)
    return p.stem   # fallback: every image is its own "writer" (risk of leakage!)

def build_metadata(root, source):
    rows, unknown = [], set()
    for p in sorted(Path(root).rglob("*")):
        if p.suffix.lower() not in IMG_EXT:
            continue
        key = norm(p.parent.name)
        if key not in LABEL_MAP:
            unknown.add(p.parent.name); continue
        rows.append(dict(path=str(p), label=LABEL_MAP[key], writer=writer_id(p), source=source))
    if unknown:
        print(f"[{source}] skipped folders not in LABEL_MAP:", sorted(unknown))
    return pd.DataFrame(rows)

df = build_metadata(DATA_DIR / "dysgraphia", "dysgraphia")
assert len(df) > 0, "No images found. Check folder names / LABEL_MAP."
print(df.label.value_counts().rename({0: "negative", 1: "positive"}))
print("Images:", len(df), "| unique writers:", df.writer.nunique())
print(df.path.map(lambda x: Path(x).name).head(10).tolist())
if df.writer.nunique() == len(df):
    print("\nWARNING: no writer IDs detected. Splits are by image, so metrics may be optimistic. "
          "Set WRITER_REGEX if filenames encode the child.")
df.to_csv(OUT_DIR / "metadata.csv", index=False)

### Theory: Images as tensors

**1. What this code does**

Shows four random images from each class and prints the sizes of a sample of images. It is a visual check before any modelling.

**2. Why it's needed**

Looking at data catches problems no metric will: wrong labels, rotated photos, images that are mostly blank paper, or one class photographed under different lighting from the other (a shortcut the model would happily learn). The size printout tells you how much the images will be shrunk later.

**3. The theory**

*A digital image is a grid of numbers.* A grayscale image is an $H \times W$ matrix; a colour image is $H \times W \times C$ with $C = 3$ channels (red, green, blue: RGB). Each value is usually an 8-bit unsigned integer (`uint8`) from 0 (black) to 255 (white).

```
Grayscale 4x4 (H=4, W=4)        RGB pixel = 3 numbers
255 255 255 255                 (R, G, B) = (255, 255, 255) white
255  20  20 255   <- ink        (R, G, B) = ( 20,  20,  20) near-black
255  20 255 255
255 255 255 255
```

*Layouts differ between libraries:*

| Library | Layout | Value range |
|---|---|---|
| PIL (Python Imaging Library, `Image`) | object; `.size` returns **(W, H)** | 0–255 |
| NumPy / matplotlib | $H \times W \times C$ | 0–255 or 0.0–1.0 |
| PyTorch | $C \times H \times W$, batched as $N \times C \times H \times W$ | float, normalised |

*Grayscale vs RGB.* Handwriting carries almost all its information in the light/dark pattern of strokes; colour mostly reflects ink and paper type. A common luminance formula is

$$Y = 0.299R + 0.587G + 0.114B$$

which weights green highest because human vision is most sensitive to it. Converting to grayscale removes colour as a possible shortcut (e.g. one site used blue pens).

**4. How it maps to the code**

- `Image.open(p).convert("L")`: `"L"` is PIL's 8-bit grayscale (luminance) mode.
- `imshow(..., cmap="gray")`: tells matplotlib to draw one channel as gray levels rather than a false-colour map.
- `df[df.label == lab].sample(4, random_state=1)`: four random images per class, repeatable via `random_state`.
- `Image.open(p).size` returns `(width, height)`: note the order.

**5. Common pitfalls**

- **(W, H) vs (H, W).** PIL reports width first, NumPy height first. Mixing them up produces transposed or distorted images.
- **Systematic differences between classes.** If all "positive" images are darker, smaller or from a different scanner, the model can reach high accuracy without looking at the handwriting. Look for this here.
- **Wide images.** Sentence images are often much wider than tall. The next phase pads them to squares, which wastes resolution (see the preprocessing theory).
- **`sample(4)` fails** if a class has fewer than 4 images; with a tiny class, lower the number.

*Deeper dive:* `theory/01_images_and_preprocessing.md`.

In [ ]:
# Look at a few samples per class
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for r, lab in enumerate([0, 1]):
    sub = df[df.label == lab].sample(4, random_state=1)
    for c, p in enumerate(sub.path):
        axes[r, c].imshow(Image.open(p).convert("L"), cmap="gray")
        axes[r, c].set_title(f"label={lab}"); axes[r, c].axis("off")
plt.tight_layout(); plt.show()
sizes = [Image.open(p).size for p in df.path.sample(min(30, len(df)), random_state=0)]
print("Sample image sizes (w,h):", sizes[:8])

## Phase 3: Writer-aware splits

`StratifiedGroupKFold` keeps all images from one writer in the same fold and keeps class ratios similar. Each fold takes a turn as the test set.

### Theory: Stratified group k-fold cross-validation

**1. What this code does**

Assigns every image to one of 5 folds so that (a) all images from one writer land in the same fold, and (b) each fold has roughly the same proportion of positive and negative images. It then checks that no writer appears in two folds and saves the table.

**2. Why it's needed**

With a few hundred images, a single train/test split leaves a test set of maybe 50 images, and the score depends heavily on *which* 50. Rotating the test set through 5 folds uses every image for testing exactly once and shows how much scores vary. The grouping prevents writer leakage; the stratification prevents a fold from accidentally having almost no positives.

**3. The theory**

*k-fold cross-validation (CV).* Split the data into $k$ disjoint folds $F_1,\dots,F_k$. For each $j$, train on the other folds and test on $F_j$. Report the mean and standard deviation of the $k$ scores:

$$\bar s = \frac{1}{k}\sum_{j=1}^{k} s_j, \qquad \mathrm{sd} = \sqrt{\frac{1}{k-1}\sum_{j=1}^{k}(s_j-\bar s)^2}$$

```
fold:     1      2      3      4      5
run 1: [TEST] [train][train][train][train]
run 2: [train][TEST] [train][train][train]
run 3: [train][train][TEST] [train][train]
run 4: [train][train][train][TEST] [train]
run 5: [train][train][train][train][TEST]
```

*Stratification* keeps the class ratio roughly constant across folds. *Grouping* treats each writer as an indivisible block. `StratifiedGroupKFold` does both, approximately: it assigns whole groups to folds while trying to balance class counts, so ratios are close but not exact.

*Why small data gives high-variance estimates.* A metric measured on $n$ test images has sampling error. For accuracy $p$ the standard error is roughly

$$\mathrm{SE} \approx \sqrt{\frac{p(1-p)}{n}}$$

With $p = 0.8$ and $n = 50$: $\sqrt{0.16/50} \approx 0.057$, so a 95% interval is about ±11 percentage points. Two models 3 points apart on 50 images are indistinguishable.

**4. How it maps to the code**

- `StratifiedGroupKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])`.
- `sgkf.split(df, df.label, df.writer)`: `y = df.label` for stratification, `groups = df.writer` for grouping. Only the test indices `te` are used to label each fold.
- `pd.crosstab(df.fold, df.label)`: check that every fold has both classes.
- `assert df.groupby("writer").fold.nunique().max() == 1`: the leakage guard.

**5. Common pitfalls**

- **The guard passes trivially when every image is its own writer.** If `WRITER_REGEX` is `None`, each "writer" has one image, so the assertion can never fail. It only protects you when real writer IDs exist.
- **Few writers per class.** If there are, say, 20 positive children, each fold gets about 4. Fold-level metrics then swing wildly; read the standard deviation, not just the mean.
- **Tuning on the test folds.** If you change the model after looking at CV results many times, the CV score itself becomes optimistic (you've indirectly fitted to it).

*Deeper dive:* `theory/05_validation_and_leakage.md`.

In [ ]:
df = df.reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
df["fold"] = -1
for i, (_, te) in enumerate(sgkf.split(df, df.label, df.writer)):
    df.loc[te, "fold"] = i
print(pd.crosstab(df.fold, df.label))

# sanity check: no writer appears in two folds
assert df.groupby("writer").fold.nunique().max() == 1
df.to_csv(OUT_DIR / "metadata_with_folds.csv", index=False)

## Phase 4: Shared modules (preprocessing + models + Grad-CAM)

These live in `LexiAssist/app/` and are **imported here** (edit them there, not in the notebook), so training and the Streamlit app use *exactly* the same preprocessing and architecture.

- Pad-to-square keeps the aspect ratio (no stroke distortion). The fill colour is taken from the image border, so it works for both white-on-black and black-on-white scans.
- Augmentation is mild (small rotation, shift, scale, brightness/contrast). **No horizontal flips**, because letter reversals may be a real signal.

### Theory: The preprocessing pipeline (`app/preprocess.py`)

**1. What this code does**

The next code cell imports three shared modules from `app/`. This first theory cell covers `preprocess.py`, which turns any uploaded photo into the exact tensor the model expects: pad to a square, resize to 224×224, copy the gray channel into 3 channels, scale to 0–1, and normalise with ImageNet statistics. The same function is used for training, evaluation and the Streamlit app.

**2. Why it's needed**

Neural networks need fixed-size numeric input. And they are very sensitive to *how* that input was produced: if training used one resize method and the app another, predictions shift even though the images "look the same". Sharing one file removes that risk (called *training–serving skew*).

**3. The theory**

*Pipeline:*

```
PIL image (any size, RGB)
   |  PadToSquare        -> grayscale, square canvas, background-coloured border
   |  Resize(224, 224)   -> bilinear interpolation
   |  Grayscale(3)       -> same gray value copied into R, G, B
   |  ToTensor           -> float tensor 3x224x224, values in [0, 1]
   v  Normalize(MEAN,STD)-> per-channel standardisation
tensor ready for the network
```

*Aspect-ratio-preserving padding.* Squashing a 1200×400 sentence straight to 224×224 would stretch letters 3× vertically, distorting exactly the stroke shapes we care about. Instead we place the image on a 1200×1200 canvas. The canvas colour is the **median of the border pixels**, a robust estimate of the paper colour that works for dark-on-light and light-on-dark scans.

*Resizing and interpolation.* Shrinking means computing new pixel values between the old ones. **Bilinear interpolation** takes a distance-weighted average of the 4 nearest pixels:

$$f(x,y) \approx \sum_{i,j \in \{0,1\}} w_{ij}\, f(x_i, y_j), \qquad w_{ij} = (1-\lvert x-x_i\rvert)(1-\lvert y-y_j\rvert)$$

When shrinking a lot, an *antialiasing* low-pass filter is applied first so fine detail blurs smoothly instead of producing jagged artefacts (*aliasing*). torchvision does this for PIL images.

*Concrete cost of resizing.* For a 1200×400 image padded to 1200×1200 and resized to 224, the scale factor is $224/1200 \approx 0.19$. The text occupies a band only about 75 pixels tall, and a 5-pixel pen stroke becomes about 1 pixel. Fine details such as tremor or small letter malformations may simply not survive.

*Why grayscale → 3 channels.* ResNet18's first layer has weights of shape $64 \times 3 \times 7 \times 7$: it expects 3 input channels. Copying gray into R, G and B lets us reuse those pretrained weights unchanged.

*Normalisation.* Each channel $c$ is standardised with the ImageNet mean $\mu_c$ and standard deviation $\sigma_c$:

$$x'_c = \frac{x_c - \mu_c}{\sigma_c}, \quad \mu = (0.485, 0.456, 0.406),\ \sigma = (0.229, 0.224, 0.225)$$

White paper ($x = 1.0$) becomes $(1-0.485)/0.229 \approx 2.25$ in the red channel; dark ink ($x = 0.08$) becomes about $-1.77$. Pretrained filters were learned on inputs with this scale, and centred inputs also make optimisation better conditioned.

**4. How it maps to the code**

| Concept | Code in `app/preprocess.py` |
|---|---|
| Median border colour | `border_fill(img)` |
| Square padding | `PadToSquare.__call__` |
| Resize | `T.Resize((img_size, img_size))` inside `build_transform` |
| 3-channel gray | `T.Grayscale(num_output_channels=3)` |
| 0–255 → 0–1 and HWC → CHW | `T.ToTensor()` |
| Standardise | `T.Normalize(MEAN, STD)` |
| Training-only augmentation | `if train: steps += [RandomAffineBG(), T.ColorJitter(...)]` |
| Image for heatmap overlays | `display_image()` (same pad + resize, no normalisation) |

**5. Common pitfalls**

- **Resolution loss.** 224×224 is small for a full sentence. If heatmaps look blurry or results are fragile, consider cropping to the text or tiling the image into patches.
- **Mismatched preprocessing.** Never re-implement these steps in the app: import them, as done here.
- **ImageNet statistics on handwriting.** Handwriting images are mostly white, so their mean is far from ImageNet's. This is fine for transfer learning (the pretrained filters expect ImageNet scaling) but worth knowing.
- **Border fill on cluttered photos.** If the photo border includes a desk or a hand, the median is not the paper colour, and the padding becomes a visible band the model might latch onto.

*Deeper dive:* `theory/01_images_and_preprocessing.md`.

### Theory: Convolutional neural networks (`SmallCNN` in `app/model.py`)

**1. What this code does**

`model.py` defines two architectures. This cell covers `SmallCNN`, a convolutional neural network (CNN) trained from scratch as a baseline: four blocks of convolution → batch normalisation → ReLU, each ending in max pooling, then global average pooling, dropout and a linear layer that outputs two scores.

**2. Why it's needed**

A plain fully-connected network on a 224×224×3 image would need about 150,000 weights *per neuron* in its first layer and would ignore the fact that nearby pixels belong together. CNNs exploit two properties of images, **locality** (strokes are local patterns) and **translation equivariance** (a loop is a loop wherever it appears), to learn with far fewer parameters. A from-scratch baseline also tells us how much the pretrained ResNet18 really adds.

**3. The theory**

*Convolution.* A small kernel (filter) $K$ slides over the image; at each position we multiply overlapping values and sum. (Deep learning libraries actually compute *cross-correlation*, without flipping the kernel, but everyone calls it convolution.)

$$Y[i,j] = \sum_{u=0}^{k-1}\sum_{v=0}^{k-1} K[u,v]\, X[i+u,\ j+v] + b$$

*Worked example:* a 5×5 image with a vertical stroke (1 = ink) and a 3×3 vertical-edge kernel:

```
X (5x5)          K (3x3)        Y = X * K (3x3, no padding, stride 1)
0 0 1 0 0        -1 0 1          3  0 -3
0 0 1 0 0        -1 0 1          3  0 -3
0 0 1 0 0        -1 0 1          3  0 -3
0 0 1 0 0
0 0 1 0 0
```

Top-left output: the window covers columns 0–2, so each row contributes $-1\cdot0 + 0\cdot0 + 1\cdot1 = 1$, total $3$. The output is **+3 at the stroke's left edge and −3 at its right edge**: the kernel is a stroke-edge detector. After ReLU only the +3 column survives. A CNN *learns* hundreds of such kernels instead of having them hand-designed.

*Stride, padding, output size.* Stride $s$ is the step between positions; padding $p$ adds a border of zeros. For input size $n$ and kernel size $k$:

$$n_{\text{out}} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1$$

Example: $n=5, k=3$: with $p=0, s=1$ → 3 (above); with $p=1, s=1$ → 5 ("same" size, used in `SmallCNN`); with $p=1, s=2$ → 3.

*Feature maps.* Each kernel produces one output channel, a *feature map* showing where its pattern occurs. A layer with 16 kernels turns a 3-channel image into a 16-channel stack.

*Pooling.* Max pooling with a 2×2 window and stride 2 keeps the largest value in each window, halving height and width. It adds a little tolerance to small shifts and reduces computation.

*Receptive field.* The region of the input that can influence one output value. Each 3×3 convolution grows it by $2 \times$ (current pixel spacing); each pooling doubles the spacing. For `SmallCNN` it grows 1 → 3 → 5 → 6 (pool) → 10 → 14 → 16 → 24 → 32 → 36 → 52 → 68 → **76 pixels**. So each position in the last feature map "sees" a 76×76 patch of the 224×224 input: roughly letter-sized, not sentence-sized. Global average pooling then averages these local detections over the whole image.

*ReLU.* $\mathrm{ReLU}(z) = \max(0, z)$. Non-linearity lets stacked layers represent more than one linear map; ReLU is cheap and its gradient is 1 for positive inputs, which helps gradients flow.

*Batch normalisation (BN).* For each channel, over a mini-batch $\mathcal B$:

$$\hat x = \frac{x - \mu_{\mathcal B}}{\sqrt{\sigma^2_{\mathcal B} + \epsilon}}, \qquad y = \gamma \hat x + \beta$$

with learnable scale $\gamma$ and shift $\beta$. It keeps activations in a stable range, allowing higher learning rates. During training it uses batch statistics and updates *running averages*; in evaluation (`model.eval()`) it uses those running averages.

*Dropout.* During training each activation is zeroed with probability $p$ and the survivors are scaled by $1/(1-p)$ so the expected value is unchanged. This stops neurons relying on specific partners and acts as a regulariser. It is switched off in `eval()` mode.

*Global average pooling (GAP).* Averages each feature map to one number, $z_k = \frac{1}{HW}\sum_{i,j} A^k_{ij}$, turning $128 \times 14 \times 14$ into a 128-vector. It has no parameters and works for any input size.

*`SmallCNN` shapes for a 224×224 input:*

```
input            3 x 224 x 224
block(3, 16)    16 x 112 x 112   conv-BN-ReLU, conv-BN-ReLU, maxpool
block(16, 32)   32 x  56 x  56
block(32, 64)   64 x  28 x  28
block(64, 128) 128 x  14 x  14
GAP            128
Dropout(0.4)   128
Linear           2   (one score, or "logit", per class)
```

Total parameters: about **295,000** (the 1.2 MB `smallcnn_fold*.pth` files are 295k × 4 bytes).

**4. How it maps to the code**

- `block(i, o)`: `nn.Conv2d(i, o, 3, padding=1)` (kernel 3, padding 1, stride 1 → same size), `nn.BatchNorm2d(o)`, `nn.ReLU(inplace=True)`, twice, then `nn.MaxPool2d(2)`.
- `self.features`: four blocks, channels 16 → 32 → 64 → 128.
- `self.classifier`: `nn.AdaptiveAvgPool2d(1)` (GAP), `nn.Flatten()`, `nn.Dropout(0.4)`, `nn.Linear(128, num_classes)`.
- `build_model("smallcnn")` returns it.

**5. Common pitfalls**

- **Forgetting `model.eval()`.** BN and dropout behave differently in training. Predicting in training mode gives noisy, batch-dependent outputs. `predict_proba` and `load_checkpoint` call `eval()`.
- **Small batches and BN.** With `batch_size=16`, batch statistics are noisy. It works here, but very small batches (2–4) can make BN unstable.
- **Expecting a from-scratch CNN to shine on a few hundred images.** It has ~295k parameters and no prior knowledge; it often overfits or underfits. That is the point of the comparison.

*Deeper dive:* `theory/02_cnns_and_resnet.md`.

### Theory: ResNet18 and residual connections

**1. What this code does**

`build_model("resnet18", pretrained=True)` loads ResNet18 with weights learned on ImageNet (1.2 million photos, 1,000 classes) and replaces its final 1,000-way layer with `Dropout(0.3)` + a 2-way linear layer. The cell also imports `explain.py`, whose Grad-CAM function hooks into ResNet18's last block (`model.layer4[-1]`); Grad-CAM theory comes in Phase 9.

**2. Why it's needed**

Deeper networks can represent richer features, but before residual networks, simply stacking more layers made training *worse*: gradients faded away and very deep plain networks had higher *training* error than shallow ones. Residual connections fixed that and made 18-, 50- and 150-layer networks trainable.

**3. The theory**

*The vanishing-gradient problem.* Backpropagation multiplies one local derivative (Jacobian) per layer:

$$\frac{\partial \mathcal L}{\partial x_0} = \frac{\partial \mathcal L}{\partial x_L}\prod_{l=1}^{L}\frac{\partial x_l}{\partial x_{l-1}}$$

If each factor has magnitude below 1, the product shrinks exponentially. With the old sigmoid activation, whose derivative is at most 0.25, ten layers give at most $0.25^{10} \approx 10^{-6}$. Early layers barely learn.

*Residual connection.* Instead of asking a block to learn the full mapping $H(x)$, let it learn a *correction* $F(x) = H(x) - x$ and add the input back:

$$y = F(x) + x \qquad\Rightarrow\qquad \frac{\partial y}{\partial x} = \frac{\partial F}{\partial x} + I$$

The identity term $I$ gives gradients a direct path backwards that is never multiplied by small factors. And if a block is not useful, it is easy for it to learn $F(x) \approx 0$ and pass the input through unchanged.

*ResNet18's basic block:*

```
        x ──────────────────────────────┐ (identity, or 1x1 conv stride 2
        │                               │  when the shape changes)
   conv 3x3 → BN → ReLU → conv 3x3 → BN │
        │                               │
        └────────────── (+) ◄───────────┘
                         │
                        ReLU
                         y
```

*The whole network for a 224×224 input:*

```
conv 7x7, stride 2   64 x 112 x 112   low-level: edges, blobs
maxpool, stride 2    64 x  56 x  56
layer1 (2 blocks)    64 x  56 x  56   strokes, corners
layer2 (2 blocks)   128 x  28 x  28   curves, simple shapes
layer3 (2 blocks)   256 x  14 x  14   parts (loops, letter fragments)
layer4 (2 blocks)   512 x   7 x   7   high-level, task-specific
GAP                 512
fc: Dropout(0.3) + Linear(512, 2)
```

"18" counts the layers with weights: 1 initial convolution + 8 blocks × 2 convolutions + 1 fully-connected layer. With a 2-class head it has about **11.2 million parameters** (hence the 45 MB checkpoints). The theoretical receptive field of `layer4` is larger than the whole image, although the *effective* receptive field (where influence is concentrated) is much smaller.

*Features at different depths.* Early layers learn generic filters (edges, gradients) that look similar in almost every image dataset, which is why they transfer well. Deep layers learn features specific to ImageNet (fur, wheels, faces), which transfer less well to handwriting. This motivates the two-stage fine-tuning in Phase 5.

**4. How it maps to the code**

- `models.resnet18(weights=models.ResNet18_Weights.DEFAULT)` when `pretrained=True`; `weights=None` in the app because the checkpoint supplies all weights.
- `m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))`: `in_features` is 512.
- Parameter names start with `conv1`, `bn1`, `layer1` … `layer4`, `fc`. `set_trainable` (Phase 5) uses the `fc` prefix to find the head.
- `explain.py`: `GradCAM(model=model, target_layers=[model.layer4[-1]])`. It only works for ResNet-style models.

**5. Common pitfalls**

- **Pretrained weights expect ImageNet preprocessing.** 3 channels, 224-ish pixels, ImageNet normalisation. That's why `preprocess.py` does exactly that.
- **11 million parameters vs a few hundred images.** Full fine-tuning can memorise the training set quickly. Freezing, a low learning rate, augmentation, dropout, weight decay and early stopping all exist to counter this.
- **`pretrained=False` in the app is correct.** Loading ImageNet weights there would be wasted work (and needs internet access): `load_state_dict` overwrites them.

*Deeper dive:* `theory/02_cnns_and_resnet.md`.

In [ ]:
assert (APP_DIR / "preprocess.py").exists(), (
    f"Shared modules not found in {APP_DIR}. Copy the project's app/ folder there "
    "(preprocess.py, model.py, explain.py) before running this cell.")

sys.path.insert(0, str(APP_DIR))
import importlib, preprocess, model as model_lib, explain
for m in (preprocess, model_lib, explain):
    importlib.reload(m)
from preprocess import build_transform, display_image, MEAN, STD
from model import build_model, load_checkpoint
from explain import gradcam_overlay
print("Shared modules ready.")

### Theory: Datasets, data loaders and augmentation

**1. What this code does**

`HandwritingDS` wraps the metadata table as a PyTorch `Dataset`: given an index, it opens the image file, applies the transform and returns `(tensor, label)`. `make_loader` wraps it in a `DataLoader` that groups samples into mini-batches. The cell then shows four augmented training images, converting them back from normalised values so you can check the strokes are intact.

**2. Why it's needed**

Images are loaded lazily, one at a time, because the full dataset may not fit in memory. Augmentation generates slightly different versions of each image every epoch. With a few hundred images, the network would otherwise memorise them; augmentation is one of the strongest defences against overfitting on small datasets.

**3. The theory**

*Dataset vs DataLoader.* A `Dataset` answers "give me item $i$". A `DataLoader` decides *which* items, in what order (`shuffle`), how many at once (`batch_size`), and loads them in parallel worker processes (`num_workers`). It stacks individual $3 \times 224 \times 224$ tensors into one $B \times 3 \times 224 \times 224$ batch.

*Data augmentation* applies random, label-preserving transformations. It encodes prior knowledge: "a slightly rotated sentence is still the same handwriting, with the same label". Formally we train on samples from $T(x)$, where $T$ is drawn from a family of transformations, so the model becomes approximately *invariant* to them.

| Type | Changes | Used here | Parameters |
|---|---|---|---|
| Geometric | Pixel *positions* | `RandomAffineBG`: rotate, shift, scale | ±5°, ±3% shift, 0.95–1.05 scale |
| Photometric | Pixel *values* | `ColorJitter`: brightness, contrast | ±20% |

An affine transform maps each pixel coordinate $(u,v)$ to

$$\begin{pmatrix}u'\\v'\end{pmatrix} = s\begin{pmatrix}\cos\phi & -\sin\phi\\ \sin\phi & \cos\phi\end{pmatrix}\begin{pmatrix}u\\v\end{pmatrix} + \begin{pmatrix}t_u\\t_v\end{pmatrix}$$

with scale $s$, angle $\phi$ and translation $t$. Areas that move into view are filled with the paper colour (`border_fill`), not black, so the model doesn't learn "black corners = augmented".

*Why no horizontal flips.* For cats and cars, a mirror image is still a cat or a car, so flips are standard. For handwriting, flipping turns `b` into `d` and makes all text run right-to-left. **Letter reversals are one of the things a dyslexia screener might care about**, so flipping would erase or invert a possible signal. Augmentations must preserve the label.

*Augment only the training data.* Validation and test images must be processed deterministically, otherwise their scores become random.

*Undoing normalisation for display:* $x = x' \cdot \sigma + \mu$, then clip to $[0,1]$.

**4. How it maps to the code**

- `HandwritingDS.__getitem__`: `Image.open(...).convert("RGB")`, then `self.tf(img)`.
- `self.tf = build_transform(CFG["img_size"], train=train)`: augmentation only when `train=True`.
- `make_loader(frame, train)`: `shuffle=train`, `batch_size=CFG["batch_size"]`, `num_workers=CFG["num_workers"]`, `drop_last=False`.
- Preview: `x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]` undoes `Normalize` (the `[:, None, None]` reshapes a 3-vector to $3\times1\times1$ so it broadcasts over $H \times W$); `.permute(1, 2, 0)` turns $C\times H\times W$ into $H\times W\times C$ for matplotlib.

**5. Common pitfalls**

- **Too-strong augmentation.** Large rotations or heavy blur may destroy exactly the features (slant, stroke quality) that distinguish the classes. Keep it mild, as here.
- **Augmentations that leak the label.** If you only augment one class, the model learns to detect augmentation.
- **Worker issues.** `num_workers > 0` can hang in some local Jupyter setups (especially on Windows); set it to 0 if loading stalls.
- **Opening files every epoch** is slow from Google Drive. For larger datasets, copy the data to the Colab local disk first.

*Deeper dive:* `theory/01_images_and_preprocessing.md`.

In [ ]:
class HandwritingDS(Dataset):
    def __init__(self, frame, train=False):
        self.paths = frame.path.tolist()
        self.labels = frame.label.tolist()
        self.tf = build_transform(CFG["img_size"], train=train)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.labels[i]

def make_loader(frame, train):
    return DataLoader(HandwritingDS(frame, train), batch_size=CFG["batch_size"],
                      shuffle=train, num_workers=CFG["num_workers"], drop_last=False)

# quick look at augmentation (train transform) to confirm strokes are preserved
ds = HandwritingDS(df.sample(4, random_state=0), train=True)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, i in zip(axes, range(4)):
    x, y = ds[i]
    img = (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0)); ax.set_title(f"label={y}"); ax.axis("off")
plt.show()

## Phase 5: Training and cross-validation

Two models, same splits:
- **SmallCNN:** custom baseline trained from scratch.
- **ResNet18:** ImageNet-pretrained, trained in two stages (head only, then full fine-tune at a low learning rate).

Class-weighted cross-entropy, AdamW, cosine schedule, early stopping on validation loss.

### Theory: Loss, backpropagation and optimisation

**1. What this code does**

This cell defines the training machinery. `run_epoch` feeds every batch through the model, measures the error with a class-weighted cross-entropy loss and, in training mode, updates the weights. `train_model` builds the loss, an AdamW optimiser and a cosine learning-rate (LR) schedule, then trains stage by stage. This first theory cell covers the loss and the optimiser; the next covers freezing, early stopping and the metrics helper.

**2. Why it's needed**

A network starts with random (or ImageNet) weights. Training is the search for weights that make the loss small on training data *without* hurting performance on new data. The loss defines what "good" means, the gradient says which direction improves it, and the optimiser decides how big a step to take.

**3. The theory**

*Softmax* turns the two raw output scores (*logits*) $z = (z_0, z_1)$ into probabilities:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

Example: $z = (1.0, 2.0)$ gives $p_1 = e^2/(e^1+e^2) \approx 0.731$, $p_0 \approx 0.269$.

*Cross-entropy* punishes low probability on the true class $y$:

$$\ell = -\log p_y$$

If $y = 1$: $\ell = -\log 0.731 \approx 0.31$. If $y = 0$: $\ell = -\log 0.269 \approx 1.31$. A confident wrong answer costs much more than a hesitant one, and $\ell \to \infty$ as $p_y \to 0$.

*Class-weighted loss.* If 60% of images are negative, a model that always predicts "negative" is already 60% accurate and receives little pressure to find positives. Weighting each example by $w_{y}$ rebalances this. The code uses

$$w_c = \frac{N}{2\,N_c}$$

so with 150 negatives and 90 positives ($N = 240$): $w_0 = 240/300 = 0.8$, $w_1 = 240/180 \approx 1.33$. PyTorch then averages as $\mathcal L = \sum_i w_{y_i}\ell_i \,/\, \sum_i w_{y_i}$.

*Backpropagation.* The chain rule computes the gradient of the loss with respect to every weight, layer by layer from the output backwards. At the output, softmax + cross-entropy has a neat gradient:

$$\frac{\partial \ell}{\partial z_k} = p_k - \mathbb 1[k = y]$$

For the example with $y = 1$: $(0.269 - 0,\ 0.731 - 1) = (0.269, -0.269)$: push $z_1$ up and $z_0$ down. `loss.backward()` carries this signal back through every layer.

*AdamW.* Plain gradient descent is $\theta \leftarrow \theta - \eta\,g$. Adam keeps running averages of the gradient (momentum) and its square (scale), giving each parameter its own effective step size:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$

$$\theta \leftarrow \theta - \eta\left(\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon} + \lambda\,\theta\right)$$

where $\hat m_t, \hat v_t$ are bias-corrected averages and the defaults are $\beta_1 = 0.9$, $\beta_2 = 0.999$. The **W** stands for *decoupled weight decay*: the $\lambda\theta$ term shrinks weights directly instead of being mixed into the adaptive gradient (in plain Adam, adding an L2 penalty to the loss gets distorted by the $\sqrt{\hat v}$ scaling). Smaller weights mean a smoother function and less overfitting. Here $\lambda = 10^{-4}$.

*Cosine schedule.* The LR starts at $\eta_0$ and decays along half a cosine to 0 over $T$ epochs:

$$\eta_t = \tfrac12\,\eta_0\left(1 + \cos\frac{\pi t}{T}\right)$$

Large steps early explore; small steps late settle into a minimum. For stage 2 of ResNet18 ($\eta_0 = 10^{-4}$, $T = 25$), the LR is $5\times10^{-5}$ at epoch 12.5 and reaches 0 at epoch 25.

**4. How it maps to the code**

| Concept | Code |
|---|---|
| Class counts → weights | `counts = train_df.label.value_counts().reindex([0, 1]).fillna(1).values`; `weights = counts.sum() / (2 * counts)` |
| Weighted cross-entropy (softmax inside) | `nn.CrossEntropyLoss(weight=weights)`: takes **logits**, not probabilities |
| Forward / backward / update | `loss = criterion(model(x), y)`; `optimizer.zero_grad(); loss.backward(); optimizer.step()` |
| No gradients during validation | `torch.set_grad_enabled(train)` |
| AdamW | `torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)` |
| Cosine schedule | `CosineAnnealingLR(opt, T_max=epochs)`; `sched.step()` once per epoch |
| Probabilities for evaluation | `predict_proba`: `torch.softmax(model(x), 1)[:, 1]` = $p_1$ |

**5. Common pitfalls**

- **Applying softmax before `CrossEntropyLoss`.** The loss applies log-softmax itself; doing it twice silently weakens training.
- **Forgetting `zero_grad()`.** PyTorch *accumulates* gradients, so without it each step uses the sum of all previous gradients.
- **Class weights change probabilities.** Weighting pushes predicted probabilities towards the minority class, so $p_1$ is not a calibrated "likelihood" (see the calibration plot in Phase 6).
- **Early stopping interrupts the schedule.** If training stops at epoch 10 of 25, the LR never decayed fully. That's fine, but it means the schedule is not doing exactly what the equation suggests.
- **Weight decay on everything.** Here AdamW also decays BN scales and biases; many practitioners exclude them. It rarely matters much at $\lambda = 10^{-4}$.

*Deeper dive:* `theory/03_training_and_optimisation.md`.

### Theory: Transfer learning, early stopping and the metrics helper

**1. What this code does**

`set_trainable` freezes or unfreezes the pretrained backbone so that only the new head (`fc` or `classifier`) learns in stage 1. `train_model` watches the validation loss after each epoch, saves a copy of the best weights, stops after `patience` epochs without improvement, and restores the best copy at the end. `compute_metrics` turns probabilities into yes/no predictions at a threshold and computes recall, specificity, precision, F1, AUC and accuracy.

**2. Why it's needed**

ResNet18 has about 11 million parameters and we have a few hundred images. Trained naively, it memorises the training set. Transfer learning starts from useful features instead of random ones; early stopping halts training at the point where it starts to memorise. Metrics then summarise how good the result is.

**3. The theory**

*Transfer learning.* ImageNet training taught ResNet18 general visual building blocks (edges, strokes, curves, textures). Learning those from a few hundred handwriting images is hopeless; reusing them leaves only the task-specific part to learn.

| Strategy | What trains | When it fits |
|---|---|---|
| Feature extraction (frozen backbone) | Head only | Very little data, or target close to ImageNet |
| Full fine-tuning | Everything | More data, or target far from ImageNet |
| **Two-stage (used here)** | Head first, then everything at a low LR | Small data, target somewhat different |

*Why two stages, and why a lower LR in stage 2.* The new head starts random, so its early gradients are large and noisy. If the backbone were trainable at that moment, those gradients would flow back and scramble the pretrained features (*catastrophic forgetting*). Stage 1 (8 epochs, LR $10^{-3}$, backbone frozen) trains the head until it is sensible. Stage 2 (up to 25 epochs, LR $10^{-4}$) then nudges *all* weights gently, staying close to the pretrained solution while adapting to handwriting.

*Overfitting, underfitting and the bias–variance trade-off.* Expected test error decomposes as

$$\mathbb E\big[(y - \hat f(x))^2\big] = \underbrace{\mathrm{Bias}^2}_{\text{too simple}} + \underbrace{\mathrm{Variance}}_{\text{too sensitive to the training set}} + \underbrace{\sigma^2}_{\text{label noise}}$$

(exact for squared error; the intuition carries over to classification).

| | Underfitting | Good fit | Overfitting |
|---|---|---|---|
| Training loss | High | Low | Very low |
| Validation loss | High | Low | Rising again |
| Main problem | Bias | | Variance |

With little data, variance dominates: the model can fit noise in a few hundred images.

*Early stopping.* Validation loss typically falls, flattens, then rises as the model starts to memorise. Keeping the weights from the minimum is a cheap, effective regulariser. `patience` tolerates noisy plateaus.

*Metrics from the confusion matrix.* Worked example with 40 test images (16 positive, 24 negative):

| | Predicted positive | Predicted negative |
|---|---|---|
| **Actually positive** | TP (true positive) = 13 | FN (false negative) = 3 |
| **Actually negative** | FP (false positive) = 6 | TN (true negative) = 18 |

| Metric | Formula | Value |
|---|---|---|
| Recall (sensitivity) | TP / (TP + FN) | 13/16 = 0.81 |
| Specificity | TN / (TN + FP) | 18/24 = 0.75 |
| Precision | TP / (TP + FP) | 13/19 = 0.68 |
| F1 | 2TP / (2TP + FP + FN) | 26/35 = 0.74 |
| Accuracy | (TP + TN) / total | 31/40 = 0.78 |

AUC (area under the ROC curve) uses the probabilities directly rather than one threshold; Phase 6 explains it.

**4. How it maps to the code**

- `set_trainable(model, trainable_backbone)`: `p.requires_grad = trainable_backbone or name.startswith(("fc", "classifier"))`.
- Stage loop: `for si, (epochs, lr, freeze) in enumerate(stages)`; `set_trainable(model, trainable_backbone=not freeze)`; a **new optimiser per stage** so it only holds trainable parameters.
- Early stopping: `if va < best_val - 1e-4` (must improve by more than $10^{-4}$), `best_state = copy.deepcopy(model.state_dict())`, `wait` counter vs `CFG["patience"]`.
- `model.load_state_dict(best_state)` restores the best epoch.
- `compute_metrics(y, p, thr=0.5)`: `pred = (p >= thr)`; `confusion_matrix(...).ravel()` returns `tn, fp, fn, tp`; `max(..., 1)` avoids division by zero.

**5. Common pitfalls**

- **"Frozen" BN layers still change.** `requires_grad=False` stops weight updates, but in `model.train()` mode BatchNorm still updates its *running mean/variance*. So the backbone is not perfectly frozen in stage 1. Usually harmless; know it exists.
- **`best_val` carries across stages.** Stage 2 must beat stage 1's best validation loss to be kept. If it never does, the final model is the stage-1 (head-only) model, which is a legitimate outcome worth noticing in the printed epoch count.
- **Early stopping on a small validation fold is noisy.** One fold ≈ 20% of a small dataset; the "best epoch" partly reflects chance.
- **Accuracy on imbalanced data.** Always read recall and specificity alongside it.
- **Validation loss that is never finite.** If it is NaN on every epoch (e.g. a corrupt image or an exploding LR), no best weights are saved. The code now raises a clear error in that case instead of crashing inside `load_state_dict`.

*Deeper dive:* `theory/04_transfer_learning.md`, `theory/03_training_and_optimisation.md`.

In [ ]:
def set_trainable(model, trainable_backbone):
    head = ("fc", "classifier")
    for name, p in model.named_parameters():
        p.requires_grad = trainable_backbone or name.startswith(head)

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = criterion(model(x), y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(y); n += len(y)
    return total / n

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(DEVICE)), 1)[:, 1].cpu().numpy()
        probs.append(p); ys.append(np.asarray(y))
    return np.concatenate(probs), np.concatenate(ys)

def train_model(model, train_df, val_df, stages, tag=""):
    model.to(DEVICE)
    counts = train_df.label.value_counts().reindex([0, 1]).fillna(1).values
    weights = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    train_loader, val_loader = make_loader(train_df, True), make_loader(val_df, False)

    best_val, best_state, history = float("inf"), None, []
    for si, (epochs, lr, freeze) in enumerate(stages):
        set_trainable(model, trainable_backbone=not freeze)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        wait = 0
        for ep in range(epochs):
            tr = run_epoch(model, train_loader, criterion, opt)
            va = run_epoch(model, val_loader, criterion)
            sched.step()
            history.append(dict(stage=si, epoch=ep, train_loss=tr, val_loss=va))
            if va < best_val - 1e-4:
                best_val, best_state, wait = va, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= CFG["patience"]:
                    break
    if best_state is None:
        raise RuntimeError(f"{tag}: validation loss never improved (NaN?); check the data and learning rate")
    model.load_state_dict(best_state)
    print(f"{tag}: best val loss {best_val:.4f} after {len(history)} epochs")
    return model, pd.DataFrame(history)

def compute_metrics(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(
        recall=tp / max(tp + fn, 1), specificity=tn / max(tn + fp, 1),
        precision=precision_score(y, pred, zero_division=0),
        f1=f1_score(y, pred, zero_division=0),
        auc=roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        accuracy=(pred == y).mean())

### Theory: The cross-validation loop and out-of-fold predictions

**1. What this code does**

For each architecture (SmallCNN, ResNet18) and each of the 5 folds, it trains a fresh model, scores the held-out test fold, records the metrics and saves the weights. Every image's test-fold probability is stored in `oof`, building a complete set of out-of-fold (OOF) predictions. It ends with a mean ± standard deviation table per architecture.

**2. Why it's needed**

This is the main experiment. It answers two questions: how well does each approach work on writers it has never seen, and how stable is that answer? Comparing a from-scratch CNN with a pretrained ResNet18 on *identical* splits makes the comparison fair.

**3. The theory**

*Three roles per run.* Each run needs data to learn from (train), data to decide when to stop (validation) and data to judge the result (test). They must be disjoint, otherwise the test score is biased: if the test fold also chose the stopping epoch, the model would be selected partly *because* it did well on that fold.

```
run k=0:  fold0=TEST  fold1=VAL   fold2,3,4=TRAIN
run k=1:  fold1=TEST  fold2=VAL   fold0,3,4=TRAIN
run k=2:  fold2=TEST  fold3=VAL   fold0,1,4=TRAIN
run k=3:  fold3=TEST  fold4=VAL   fold0,1,2=TRAIN
run k=4:  fold4=TEST  fold0=VAL   fold1,2,3=TRAIN
```

Each model trains on 60% of the data, early-stops on 20% and is tested on 20%.

*Out-of-fold predictions.* After all 5 runs, every image has exactly one probability, produced by a model that never saw that image (or its writer, if writer IDs are real) during training or stopping. Pooling them gives one prediction per image for the whole dataset:

$$\hat p_i = f_{\theta^{(-k(i))}}(x_i)$$

where $k(i)$ is image $i$'s fold and $\theta^{(-k)}$ the weights trained without fold $k$. Phase 6 builds ROC curves, calibration plots and the threshold from these.

*Reading mean ± sd.* The mean estimates expected performance; the standard deviation across folds shows how much it depends on *which* writers you test on. With a few hundred images, a standard deviation of 0.05–0.10 on recall is normal. If two models' means are closer than their standard deviations, you cannot claim one is better.

**4. How it maps to the code**

- `ARCHS = {"smallcnn": CFG["stages_cnn"], "resnet18": CFG["stages_resnet"]}`: SmallCNN trains in a single 40-epoch stage; ResNet18 in the two transfer stages.
- `test_df = df[df.fold == k]`, `val_df = df[df.fold == (k + 1) % CFG["n_folds"]]`, `train_df` = the rest.
- `set_seed(CFG["seed"] + k)`: different but reproducible randomness per fold.
- `build_model(arch, pretrained=(arch == "resnet18"))`: only ResNet18 loads ImageNet weights.
- `oof[arch].loc[test_df.index, "prob"] = p`: fills in OOF probabilities.
- `compute_metrics(y, p)` at the default threshold 0.5 → `fold_rows` → `cv_fold_metrics.csv`.
- `torch.save(model.state_dict(), CKPT_DIR / f"{arch}_fold{k}.pth")`: weights only (no metadata), saved from the GPU.

**5. Common pitfalls**

- **The 5 fold models are not "the model".** CV estimates how well *the procedure* works. The deployed model is trained separately in Phase 7.
- **Metrics at 0.5.** These fold metrics use 0.5; the screening threshold chosen in Phase 6 will trade specificity for recall.
- **Pooled vs averaged metrics differ.** AUC computed on pooled OOF predictions (Phase 6) is not the same as the mean of per-fold AUCs, because each fold model has its own probability scale.
- **Loading fold checkpoints on a CPU** needs `torch.load(path, map_location="cpu")`, because they were saved from `cuda:0`.

*Deeper dive:* `theory/05_validation_and_leakage.md`.

In [ ]:
ARCHS = {"smallcnn": CFG["stages_cnn"], "resnet18": CFG["stages_resnet"]}
oof = {a: pd.DataFrame({"idx": df.index, "label": df.label, "prob": np.nan}).set_index("idx") for a in ARCHS}
fold_rows, histories = [], {}

for arch, stages in ARCHS.items():
    for k in range(CFG["n_folds"]):
        set_seed(CFG["seed"] + k)
        test_df = df[df.fold == k]
        val_df  = df[df.fold == (k + 1) % CFG["n_folds"]]
        train_df = df[~df.fold.isin([k, (k + 1) % CFG["n_folds"]])]

        model = build_model(arch, pretrained=(arch == "resnet18"))
        model, hist = train_model(model, train_df, val_df, stages, tag=f"{arch} fold {k}")
        histories[(arch, k)] = hist

        p, y = predict_proba(model, make_loader(test_df, False))
        oof[arch].loc[test_df.index, "prob"] = p
        fold_rows.append(dict(arch=arch, fold=k, **compute_metrics(y, p)))
        torch.save(model.state_dict(), CKPT_DIR / f"{arch}_fold{k}.pth")

fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(OUT_DIR / "cv_fold_metrics.csv", index=False)
summary = fold_df.groupby("arch")[["recall", "specificity", "precision", "f1", "auc", "accuracy"]].agg(["mean", "std"]).round(3)
summary

### Theory: Reading learning curves

**1. What this code does**

Plots training loss and validation loss per epoch for fold 0 of each architecture, using the `history` that `train_model` recorded.

**2. Why it's needed**

Final metrics tell you *how well* a model did; learning curves tell you *why*. They are the quickest way to diagnose underfitting, overfitting, a bad learning rate or a stage transition that went wrong.

**3. The theory**

Training loss measures fit to data the model learns from; validation loss estimates fit to unseen data. The gap between them is the **generalisation gap**.

```
Underfitting                Good fit                    Overfitting
loss                        loss                        loss
 |\_____ val                 |\                          |\        ___ val
 | \____ train               | \___ val                  | \_____/
 |                           |  \___ train               |  \
 |                           |                           |   \______ train
 +-------- epoch             +-------- epoch             +-------- epoch
both high, flat             both low, small gap         train keeps falling,
-> model too weak or        -> stop around here         val turns upward
   LR too low                                           -> early stopping point
```

Other patterns:

| Pattern | Likely cause |
|---|---|
| Validation loss jumps around a lot | Small validation set; noisy estimate |
| Loss explodes or becomes NaN | LR too high, or a corrupt input |
| Visible kink in ResNet18's curve | Stage 1 → 2 transition: backbone unfreezes and LR drops |
| Validation loss *below* training loss | Dropout and augmentation are active in training but not validation; often normal early on |

**4. How it maps to the code**

- `histories[(arch, 0)]`: one row per epoch, columns `stage`, `epoch`, `train_loss`, `val_loss`.
- `h.train_loss.values` / `h.val_loss.values` plotted against the epoch index. ResNet18's x-axis spans both stages, so the stage-2 epochs follow straight after stage 1's.
- The curves stop where early stopping fired; the weights actually kept are those at the lowest `val_loss`, not the last epoch.

**5. Common pitfalls**

- **Over-reading one fold.** Fold 0 is one sample; check others (`histories[(arch, k)]`) before concluding.
- **Loss isn't the goal.** Validation loss can rise slightly while recall improves. Loss is a smooth proxy used for stopping; the metrics are what you report.
- **Comparing absolute losses across models.** Dropout rates (0.4 vs 0.3), capacity and training stages differ between the architectures, so compare shapes and gaps more than raw values.

*Deeper dive:* `theory/03_training_and_optimisation.md`.

In [ ]:
# Training curves for one fold (check for overfitting: widening train/val gap)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, arch in zip(axes, ARCHS):
    h = histories[(arch, 0)]
    ax.plot(h.train_loss.values, label="train"); ax.plot(h.val_loss.values, label="val")
    ax.set_title(f"{arch} (fold 0)"); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()

## Phase 6: Evaluation and threshold

Out-of-fold predictions mean every image is scored by a model that never saw its writer. For a screening tool, **recall (sensitivity)** matters most, but watch specificity too, since false alarms upset people.

### Theory: ROC curves, AUC, the confusion matrix and calibration

**1. What this code does**

Draws three plots from the out-of-fold predictions: ROC curves for both architectures with their AUC, the confusion matrix for ResNet18 at threshold 0.5, and a calibration (reliability) curve for ResNet18. The figure is saved to `outputs/evaluation.png`.

**2. Why it's needed**

A single accuracy number hides the trade-off that matters for screening: catching more children with real difficulties (recall) always costs more false alarms (lower specificity). The ROC curve shows that whole trade-off; the confusion matrix makes one operating point concrete; calibration tells you whether "70%" actually means 70%.

**3. The theory**

*ROC (receiver operating characteristic) curve.* Sweep the threshold $t$ from 1 down to 0. At each $t$, compute the true positive rate (TPR, = recall) and false positive rate (FPR, = 1 − specificity):

$$\mathrm{TPR}(t) = \frac{TP(t)}{TP(t)+FN(t)}, \qquad \mathrm{FPR}(t) = \frac{FP(t)}{FP(t)+TN(t)}$$

and plot TPR against FPR. A useless model follows the diagonal; a perfect one hugs the top-left corner.

*AUC (area under the curve)* has a clean meaning: **the probability that a randomly chosen positive image gets a higher score than a randomly chosen negative one.**

$$\mathrm{AUC} = P(\hat p_{+} > \hat p_{-})$$

Worked example: positive scores $\{0.9, 0.6, 0.4\}$, negative scores $\{0.7, 0.3, 0.2\}$. Of the $3\times3 = 9$ positive–negative pairs, the positive wins in 3 (for 0.9) + 2 (for 0.6) + 2 (for 0.4) = 7, so AUC = 7/9 ≈ 0.78. AUC ignores the threshold and the probability scale: it only measures *ranking*.

| AUC | Rough reading |
|---|---|
| 0.5 | No better than chance |
| 0.7–0.8 | Useful but weak |
| 0.8–0.9 | Good |
| > 0.95 on a small dataset | Be suspicious: check for leakage or shortcuts |

*Confusion matrix.* The 2×2 table of TP, FN, FP, TN at one threshold (see the worked example in the Phase 5 theory). Rows are true labels, columns predicted labels.

*Calibration.* A model is *calibrated* if, among all images it scores about 0.7, about 70% are truly positive. The reliability curve groups predictions into bins and plots the observed fraction of positives against the mean predicted probability; perfect calibration is the diagonal. One summary is the expected calibration error (ECE):

$$\mathrm{ECE} = \sum_{b=1}^{B}\frac{n_b}{n}\,\big\lvert \mathrm{acc}_b - \mathrm{conf}_b \big\rvert$$

Points below the diagonal mean over-confidence; above it, under-confidence.

**4. How it maps to the code**

- `roc_curve(d.label, d.prob)` returns `fpr, tpr, thresholds`; `roc_auc_score` gives the area.
- `confusion_matrix(d.label, d.prob >= 0.5, labels=[0, 1])` with `ConfusionMatrixDisplay(..., display_labels=["negative", "positive"])`.
- `calibration_curve(d.label, d.prob, n_bins=5, strategy="quantile")`: 5 bins with equal *numbers* of images (not equal width), so each point rests on the same amount of data.
- `d = oof["resnet18"]`: everything here uses pooled OOF predictions.

**5. Common pitfalls**

- **AUC is not accuracy.** A model with AUC 0.85 can still have poor recall at a badly chosen threshold.
- **Pooling across folds.** Each fold model has its own probability scale; pooling their OOF scores can slightly blur both ROC and calibration.
- **Few bins, few images.** With a few hundred images, each of 5 calibration bins holds only dozens of images; the curve is a rough sketch with wide error bars, not a precise measurement.
- **Class weighting and calibration.** Weighted training shifts probabilities upward for the minority class, so miscalibration here is expected. The app's "estimated likelihood" percentage should be read as a score, not a true probability.

*Deeper dive:* `theory/06_metrics_and_thresholds.md`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# ROC curves (out-of-fold)
for arch in ARCHS:
    d = oof[arch]
    fpr, tpr, _ = roc_curve(d.label, d.prob)
    axes[0].plot(fpr, tpr, label=f"{arch} (AUC {roc_auc_score(d.label, d.prob):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("Recall"); axes[0].legend(); axes[0].set_title("ROC (out-of-fold)")

# Confusion matrix for ResNet18 at 0.5
d = oof["resnet18"]
ConfusionMatrixDisplay(confusion_matrix(d.label, d.prob >= 0.5, labels=[0, 1]),
                       display_labels=["negative", "positive"]).plot(ax=axes[1], colorbar=False)
axes[1].set_title("ResNet18 confusion matrix @0.5")

# Calibration
frac, mean_pred = calibration_curve(d.label, d.prob, n_bins=5, strategy="quantile")
axes[2].plot(mean_pred, frac, "o-"); axes[2].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[2].set_xlabel("Predicted probability"); axes[2].set_ylabel("Observed fraction positive")
axes[2].set_title("Calibration (ResNet18)")
plt.tight_layout(); plt.savefig(OUT_DIR / "evaluation.png", dpi=150); plt.show()

### Theory: Choosing a screening threshold

**1. What this code does**

Finds the decision threshold that gives the highest specificity while keeping recall at or above 90% on the ResNet18 out-of-fold predictions, clips it to the range 0.05–0.95, and prints the full metric set at that threshold. This `THRESHOLD` is saved with the final model and used by the app.

**2. Why it's needed**

The default threshold of 0.5 is arbitrary. What matters is the *cost* of each kind of mistake, and for a screening aid those costs are very unequal.

**3. The theory**

*Two kinds of error:*

| Error | Meaning here | Consequence |
|---|---|---|
| False negative (FN) | Child with difficulties told "few indicators" | **False reassurance**: support may be delayed |
| False positive (FP) | Child without difficulties told "possible indicators" | **False alarm**: worry, plus an unnecessary specialist referral |

Screening tools usually favour **recall (sensitivity)**: a missed case is often costlier than a referral that turns out unnecessary, because the next step (a specialist assessment) catches false alarms. That is why the rule here is "recall ≥ 0.90 first, then the best specificity".

*Formally*, choose

$$t^* = \arg\max_t\ \mathrm{Specificity}(t) \quad \text{subject to} \quad \mathrm{Recall}(t) \ge 0.90$$

Alternatives include maximising Youden's $J = \text{recall} + \text{specificity} - 1$ (treats both errors equally) or minimising an explicit cost $c_{FN}\cdot FN + c_{FP}\cdot FP$.

*Prevalence changes what a positive result means.* Precision, also called PPV (positive predictive value), depends on how common the condition is, not just on the model. Suppose recall 0.90, specificity 0.70, and 10% of children truly have difficulties. Per 1,000 children:

| | Flagged | Not flagged | Total |
|---|---|---|---|
| Have difficulties | 90 | 10 | 100 |
| Don't | 270 | 630 | 900 |

$$\mathrm{PPV} = \frac{90}{90+270} = 25\%, \qquad \mathrm{NPV} = \frac{630}{630+10} \approx 98.4\%$$

(NPV: negative predictive value.) **Three out of four flagged children would be false alarms**, even with a decent model. The class balance in the training data is usually much higher than real-world prevalence, so precision measured in this notebook overstates real-world precision.

**4. How it maps to the code**

- `TARGET_RECALL = 0.90`.
- `fpr, tpr, thr = roc_curve(d.label, d.prob)`: every candidate threshold with its rates.
- `ok = np.where(tpr >= TARGET_RECALL)[0]`: thresholds meeting the recall target.
- `best = ok[np.argmin(fpr[ok])]`: among those, lowest FPR = highest specificity.
- `THRESHOLD = float(np.clip(thr[best], 0.05, 0.95))`: keeps the threshold away from extremes.
- `compute_metrics(d.label.values, d.prob.values, THRESHOLD)`: metrics at the final threshold.

**5. Common pitfalls**

- **Optimistic numbers.** The threshold is chosen on the same OOF predictions it is evaluated on, so recall ≥ 0.90 is guaranteed *on this data* and will likely be lower on new children. A cleaner approach is nested CV or a held-out calibration set.
- **The threshold belongs to different models.** It was tuned on the 5 CV models' scores but is applied to the final model from Phase 7, whose probability scale may differ.
- **Clipping can move the operating point.** If the raw threshold falls outside 0.05–0.95, clipping changes it. The first printed line shows recall/specificity *before* clipping; trust the second printout, computed at the clipped `THRESHOLD`.
- **0.90 is a choice, not a law.** It should come from the setting: who acts on the result and what a referral costs. State and justify it in your write-up.

*Deeper dive:* `theory/06_metrics_and_thresholds.md`.

In [ ]:
# Screening threshold: highest specificity that still reaches the target recall
TARGET_RECALL = 0.90
d = oof["resnet18"]
fpr, tpr, thr = roc_curve(d.label, d.prob)
ok = np.where(tpr >= TARGET_RECALL)[0]
best = ok[np.argmin(fpr[ok])]
THRESHOLD = float(np.clip(thr[best], 0.05, 0.95))
print(f"Chosen threshold: {THRESHOLD:.3f}  (recall {tpr[best]:.2f}, specificity {1 - fpr[best]:.2f})")
print(pd.Series(compute_metrics(d.label.values, d.prob.values, THRESHOLD)).round(3))
# Note: the threshold is tuned on the same out-of-fold predictions, so treat these numbers as slightly optimistic.

## Phase 7: Final model (for the app)

Cross-validation estimates performance. For deployment, train one ResNet18 on all folds except a validation fold (used only for early stopping), then save it with everything the app needs.

### Theory: Training the deployment model

**1. What this code does**

Trains one more ResNet18 on folds 0, 2, 3 and 4, using fold 1 only for early stopping. It then saves a checkpoint containing the weights *and* everything the app needs to use them: architecture name, class names, image size, normalisation statistics, the screening threshold and a description of the training data. The file is copied into `app/`.

**2. Why it's needed**

Cross-validation produced five models, each trained on only 60% of the data. For deployment we want one model trained on as much data as possible. Packaging the metadata with the weights means the app can't accidentally use a different image size or threshold from the one the model was evaluated with.

**3. The theory**

*Estimate vs artefact.* CV estimates how well *the training procedure* generalises. Retraining on more data with the same procedure usually gives a model at least as good, so the CV estimate is used as a (slightly pessimistic) estimate for the final model. **The final model has no independent test score of its own**: every image was used for training or early stopping.

*Why still hold out a validation fold?* Early stopping needs data the model doesn't train on. Using fold 1 for this means the final model trains on 80% of the data instead of 100%. An alternative is to retrain on everything for a fixed number of epochs (e.g. the median best epoch from CV).

*What a checkpoint should contain.* A model is not just weights: it is weights + architecture + preprocessing + decision rule. Storing them together is the cheapest defence against training–serving skew.

| Key | Why the app needs it |
|---|---|
| `state_dict` | The learned weights |
| `arch`, `class_names` | Rebuild the right network with the right number of outputs |
| `img_size`, `mean`, `std` | Reproduce the exact preprocessing |
| `threshold` | Same decision rule as evaluated |
| `trained_on` | Honest provenance shown to users |

**4. How it maps to the code**

- `VAL_FOLD = 1`; `final_train = df[df.fold != VAL_FOLD]`.
- `train_model(final_model, final_train, final_val, CFG["stages_resnet"], tag="final resnet18")`: same two-stage recipe as CV.
- `ckpt = dict(state_dict=final_model.cpu().state_dict(), arch="resnet18", ...)`: moved to the CPU first, so the app can load it without a GPU.
- `torch.save(ckpt, CKPT_DIR / "lexiassist.pth")` then `shutil.copy(...)` into `APP_DIR`.
- `app/model.py: load_checkpoint` reads exactly these keys.

**5. Common pitfalls**

- **The threshold came from different models** (see Phase 6). The final model's scores may sit slightly higher or lower; ideally re-check the threshold on fresh data.
- **Phase 9 uses `final_val` for explanations.** Fold 1 was used for early stopping, so it is not a pure test set, but it was not used for gradient updates.
- **Checkpoints of children's-data models.** The weights may memorise aspects of training images. Treat the `.pth` as derived from sensitive data when deciding where to publish it.
- **Saving only `state_dict`** (as the fold checkpoints do) loses the metadata; you then have to remember the architecture and preprocessing yourself.

*Deeper dive:* `theory/05_validation_and_leakage.md`.

In [ ]:
set_seed(CFG["seed"])
VAL_FOLD = 1
final_val   = df[df.fold == VAL_FOLD]
final_train = df[df.fold != VAL_FOLD]

final_model = build_model("resnet18", pretrained=True)
final_model, final_hist = train_model(final_model, final_train, final_val, CFG["stages_resnet"], tag="final resnet18")

CLASS_NAMES = ["Low indicators", "Possible indicators"]
ckpt = dict(
    state_dict=final_model.cpu().state_dict(), arch="resnet18", class_names=CLASS_NAMES,
    img_size=CFG["img_size"], mean=MEAN, std=STD, threshold=THRESHOLD,
    trained_on="Potential Dysgraphia Handwriting Dataset (Malay sentences, children)",
)
torch.save(ckpt, CKPT_DIR / "lexiassist.pth")
shutil.copy(CKPT_DIR / "lexiassist.pth", APP_DIR / "lexiassist.pth")
final_model.to(DEVICE).eval()
print("Saved:", APP_DIR / "lexiassist.pth")

## Phase 8: External test (different source)

Run the final model on the bilingual dataset. It differs in script, task (characters vs sentences) and labels (dyslexia vs potential dysgraphia), so **expect a drop**. A large drop shows the model learned dataset-specific features, which is a valid and useful finding for your write-up.

### Theory: External validation and dataset shift

**1. What this code does**

If the bilingual dataset is present, builds its metadata with the same `LABEL_MAP`, runs the final model on every image, and prints the metrics at threshold 0.5 and at the chosen screening threshold.

**2. Why it's needed**

Cross-validation tests on new *writers* from the *same* dataset: same school system, same task, same cameras, same assessors. Real users bring different everything. External validation is the closest this notebook gets to asking "does this work in the real world?". A large drop is a genuine, reportable finding, not a failure of the notebook.

**3. The theory**

A model learns patterns in the training distribution $P_{\text{train}}(x, y)$. *Dataset shift* means the deployment distribution $P_{\text{test}}(x, y)$ differs. Writing $P(x, y) = P(y \mid x)\,P(x)$ separates three kinds:

| Shift | What changes | Example here |
|---|---|---|
| Covariate shift | $P(x)$: what images look like | Different script, characters vs sentences, cameras, paper |
| Label (prior) shift | $P(y)$: how common positives are | Different proportion of positives |
| Concept shift | $P(y \mid x)$: what the label *means* | **Dyslexia** labels vs **potential dysgraphia** labels |

Concept shift is the most serious here: the two datasets don't even measure the same condition, so some performance drop is unavoidable.

*Shortcut learning.* Models take the easiest route to low loss. If, in the training set, positive samples happen to come from a particular classroom with darker paper, "dark paper → positive" is a perfectly good rule *on that dataset* and useless elsewhere. External data exposes shortcuts because the accidental correlations are different there.

```
training data:  handwriting features + site/camera/paper quirks  -> label
                         ^ what we want          ^ what's easy
external data:  quirks are different  ->  shortcut rules break  ->  metrics drop
```

**4. How it maps to the code**

- `ext_dir = DATA_DIR / "bilingual"`; `build_metadata(ext_dir, "bilingual")` reuses the same folder-name labelling, so its folders must match `LABEL_MAP` keys (e.g. `dyslexic` / `non dyslexic`).
- `predict_proba(final_model, make_loader(ext, False))`: deterministic preprocessing, no augmentation.
- Metrics are printed at `0.5` and at `THRESHOLD`.

**5. Common pitfalls**

- **Tuning on the external set.** Once you change anything because of these results, the external set is no longer independent. Look once, report honestly.
- **Different task, different image type.** Character-level images padded to squares look nothing like sentence images; the model has never seen such inputs.
- **Small external sets.** The same sampling-error caveat as before: report counts alongside percentages.
- **Overclaiming either way.** A drop shows the model doesn't transfer to *that* dataset; it doesn't prove the model learned nothing. Good external performance on one dataset doesn't prove it works across languages in general.

*Deeper dive:* `theory/08_responsible_ai.md`.

In [ ]:
ext_dir = DATA_DIR / "bilingual"
if ext_dir.exists():
    ext = build_metadata(ext_dir, "bilingual")
    if len(ext):
        p, y = predict_proba(final_model, make_loader(ext, False))
        print(f"External images: {len(ext)}")
        print("At 0.5      :", {k: round(v, 3) for k, v in compute_metrics(y, p, 0.5).items()})
        print(f"At {THRESHOLD:.2f} :", {k: round(v, 3) for k, v in compute_metrics(y, p, THRESHOLD).items()})
    else:
        print("No labelled images found in", ext_dir, "- check folder names / LABEL_MAP.")
else:
    print("Skipping: put the bilingual dataset in", ext_dir)

## Phase 9: Explainability (Grad-CAM + Integrated Gradients)

**Sanity check:** do the heatmaps sit on the pen strokes? If they highlight paper edges, shadows or ruled lines, the model is using shortcuts. In that case, crop or binarize the images and retrain.

### Theory: Grad-CAM

**1. What this code does**

For six validation images, `show_gradcam` computes the final model's probability, then a Grad-CAM (gradient-weighted class activation mapping) heatmap for class 1 ("possible indicators"), and shows the processed image above its heatmap overlay. The figure is saved to `outputs/gradcam_examples.png`. The app uses the same `gradcam_overlay` function.

**2. Why it's needed**

A classifier can be right for the wrong reasons. Heatmaps are a quick check of *where* the evidence came from: pen strokes (encouraging) or paper edges, shadows, ruled lines or padding (a warning sign of shortcut learning).

**3. The theory**

*Intuition.* The last convolutional block (`layer4`) holds 512 feature maps of size 7×7, each a coarse "where is pattern $k$?" map. Grad-CAM asks: *which of these maps, if they got stronger, would raise the score for class $c$?* It weights each map by that importance and adds them up.

*Step 1: channel importance.* Let $y^c$ be the class-$c$ logit (score before softmax) and $A^k$ the $k$-th feature map. Average the gradient over all positions:

$$\alpha_k^c = \frac{1}{Z}\sum_{i}\sum_{j}\frac{\partial y^c}{\partial A^k_{ij}}, \qquad Z = 7 \times 7$$

This is global average pooling applied to the *gradients*.

*Step 2: weighted sum, keep the positive part:*

$$L^c_{\text{Grad-CAM}} = \mathrm{ReLU}\Big(\sum_k \alpha_k^c A^k\Big)$$

The ReLU keeps only regions that push the class-$c$ score *up*. The 7×7 map is then normalised to $[0,1]$, upsampled to 224×224 and blended over the image.

```
image -> conv layers -> layer4 feature maps A^k (512 x 7 x 7) -> GAP -> fc -> y^c
                               |                                             |
                               |  <------- gradients dy^c/dA^k --------------+
                               v
              alpha_k = mean of gradients  ->  ReLU(sum_k alpha_k A^k)  ->  7x7 heatmap
```

*Resolution.* Each heatmap cell covers about $224/7 = 32$ pixels square, roughly a whole letter or two after resizing. Grad-CAM can say "this word region", never "this stroke".

*What saliency maps can and cannot tell you:*

| Can | Cannot |
|---|---|
| Show where the score is most sensitive for this image | Prove the model uses clinically meaningful features |
| Reveal obvious shortcuts (edges, shadows, padding) | Explain *what* about a region mattered (slant? spacing? size?) |
| Support a qualitative sanity check | Give a stable, precise or causal explanation |

Some saliency methods have been shown to produce similar maps even for a network with randomised weights (Adebayo et al., 2018, "Sanity checks for saliency maps"). Grad-CAM passes those checks better than many methods, but heatmaps are still evidence to look at, not proof.

**4. How it maps to the code**

- `app/explain.py: gradcam_overlay(model, x, rgb_img, target_class=1)`
  - `GradCAM(model=model, target_layers=[model.layer4[-1]])`: $A^k$ comes from the last basic block.
  - `ClassifierOutputTarget(target_class)`: selects $y^c$, the class-1 logit.
  - `show_cam_on_image(rgb_img, heat, use_rgb=True)`: colour-maps and blends the heatmap.
- `display_image(img, CFG["img_size"])`: the padded, resized image *without* normalisation, so the overlay lines up with what the model saw.
- `axes[0, j].set_title(f"true={row.label}  p={prob:.2f}")`: compare the heatmap with the true label and the score.

**5. Common pitfalls**

- **Always explaining class 1.** The heatmap shows evidence *for* "possible indicators", even when the model predicts "low indicators". For a negative prediction, a weak or scattered heatmap is expected.
- **Highlighting padding.** Bright regions in the padded border mean the model may be reacting to image *shape* (e.g. sentence length) rather than handwriting.
- **Cherry-picking.** Six random images are a spot check. Look at many, including false positives and false negatives.
- **Over-trusting colours.** The map is normalised per image, so "red" means "most important *in this image*", not "important" in absolute terms.

*Deeper dive:* `theory/07_explainability.md`.

In [ ]:
def show_gradcam(frame, n=6, seed=0):
    sub = frame.sample(min(n, len(frame)), random_state=seed)
    tf = build_transform(CFG["img_size"], train=False)
    fig, axes = plt.subplots(2, len(sub), figsize=(3.2 * len(sub), 6.4))
    axes = np.atleast_2d(axes).reshape(2, -1)
    for j, (_, row) in enumerate(sub.iterrows()):
        img = Image.open(row.path).convert("RGB")
        x = tf(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prob = torch.softmax(final_model(x), 1)[0, 1].item()
        rgb = display_image(img, CFG["img_size"])
        overlay = gradcam_overlay(final_model, x, rgb, target_class=1)
        axes[0, j].imshow(rgb); axes[0, j].set_title(f"true={row.label}  p={prob:.2f}", fontsize=9)
        axes[1, j].imshow(overlay)
        for i in (0, 1): axes[i, j].axis("off")
    plt.tight_layout(); plt.savefig(OUT_DIR / "gradcam_examples.png", dpi=150); plt.show()

# Use images from the validation fold (not used for gradient updates)
show_gradcam(final_val, n=6)

### Theory: Integrated Gradients

**1. What this code does**

For four validation images, computes Integrated Gradients (IG) attributions for class 1 using the Captum library: a per-pixel score of how much each pixel contributed to the class-1 logit, relative to a baseline image. It shows each image above its attribution map.

**2. Why it's needed**

Grad-CAM is coarse (7×7) and depends on one layer. IG works directly at pixel resolution and has a mathematical guarantee (completeness). If two very different methods highlight the same regions, the explanation is more believable; if they disagree, be cautious about both.

**3. The theory**

*Why not just use the gradient?* The plain gradient $\partial F / \partial x_i$ only describes a tiny neighbourhood of the input. Once a feature is "saturated" (e.g. a ReLU unit is already fully on), its gradient can be zero even though the feature matters a lot.

*Idea.* Start from a **baseline** $x'$ that represents "no information", walk in a straight line to the real image $x$, and add up the gradients along the way:

$$\mathrm{IG}_i(x) = (x_i - x'_i)\int_{0}^{1}\frac{\partial F\big(x' + \alpha(x - x')\big)}{\partial x_i}\,d\alpha$$

*In practice* the integral is approximated with $m$ points (a Riemann-style sum; Captum uses Gauss–Legendre points by default):

$$\mathrm{IG}_i(x) \approx (x_i - x'_i)\sum_{k=1}^{m} w_k\,\frac{\partial F\big(x' + \alpha_k(x - x')\big)}{\partial x_i}$$

Here $m$ = `n_steps` = 40, so each image needs 40 forward and backward passes.

*Completeness axiom.* The attributions add up exactly to the change in output:

$$\sum_i \mathrm{IG}_i(x) = F(x) - F(x')$$

So IG divides the model's score (relative to the baseline) among the pixels. Captum can report how far the approximation is from this (`return_convergence_delta=True`); a large gap means `n_steps` is too small.

```
baseline x'  ---- alpha = 0.25 ---- 0.5 ---- 0.75 ---->  image x
   (gradients measured at each point, averaged, times (x - x'))
```

**4. How it maps to the code**

- `IntegratedGradients(final_model)`: $F$ = the model's output; `target=1` selects the class-1 **logit** (the model returns logits, not probabilities).
- `baselines=torch.zeros_like(x)`: $x' = 0$ *in normalised space*.
- `n_steps=40`: $m$.
- `attr.abs().sum(1)[0]`: absolute value, summed over the 3 (identical gray) channels → one 224×224 map; then scaled to $[0,1]$ for display.

**5. Common pitfalls**

- **The baseline is not white paper.** After ImageNet normalisation, an all-zero tensor means "every pixel equals the ImageNet mean", a mid-gray (about 0.45 brightness). Paper is near white, so blank paper differs from the baseline and can receive attribution too. A white-paper baseline (the normalised value of 1.0) or a blurred copy of the image are common alternatives; the choice changes the result, so state it.
- **`abs()` hides direction.** Positive attributions support class 1, negative ones oppose it. Taking the absolute value shows *importance* but not *which way*.
- **Speckled maps.** Pixel-level attributions are noisy; smoothing or averaging over several baselines makes them easier to read.
- **Same limits as Grad-CAM.** IG explains sensitivity of this model's logit, not whether its reasoning is clinically valid.

*Deeper dive:* `theory/07_explainability.md`.

In [ ]:
# Second method for cross-checking: Integrated Gradients (Captum)
from captum.attr import IntegratedGradients

def show_ig(frame, n=4, seed=1):
    sub = frame.sample(min(n, len(frame)), random_state=seed)
    tf = build_transform(CFG["img_size"], train=False)
    ig = IntegratedGradients(final_model)
    fig, axes = plt.subplots(2, len(sub), figsize=(3.2 * len(sub), 6.4))
    axes = np.atleast_2d(axes).reshape(2, -1)
    for j, (_, row) in enumerate(sub.iterrows()):
        img = Image.open(row.path).convert("RGB")
        x = tf(img).unsqueeze(0).to(DEVICE).requires_grad_(True)
        attr = ig.attribute(x, baselines=torch.zeros_like(x), target=1, n_steps=40)
        heat = attr.abs().sum(1)[0].detach().cpu().numpy()
        heat = heat / (heat.max() + 1e-8)
        axes[0, j].imshow(display_image(img, CFG["img_size"])); axes[1, j].imshow(heat, cmap="inferno")
        axes[0, j].set_title(f"true={row.label}", fontsize=9)
        for i in (0, 1): axes[i, j].axis("off")
    plt.tight_layout(); plt.show()

show_ig(final_val, n=4)

### Theory: Responsible AI

There is no code in this section; it pulls together the ethical and practical limits that run through the whole notebook. Read it before using or presenting the app.

**1. What this project actually does**

It ranks photos of handwriting by how much they resemble the "potential dysgraphia" examples in one small dataset of Malay-speaking children. That is all. It does not detect dyslexia, and it cannot be used to diagnose anything.

**2. Why it matters**

Tools that sound medical get trusted more than they deserve. A parent who sees "18%" may stop worrying about a child who needs support; a teacher who sees "82%" may label a child who is fine. Being clear about limits is part of building the tool, not an add-on.

**3. The key ideas**

*Screening vs diagnosis.*

| | Screening | Diagnosis |
|---|---|---|
| Purpose | Decide who should be looked at more closely | Decide whether a condition is present |
| Done by | Teachers, questionnaires, simple tools | Qualified specialists (e.g. educational psychologists) |
| Errors | Expected; the next step catches them | Must be minimised; decisions follow from it |
| LexiAssist | At most a weak, experimental screening aid | **Not this** |

*The two failure modes carry different harms.* False reassurance (FN) can delay support; false alarms (FP) cause worry and unnecessary referrals. The threshold encodes a value judgement about which harm is worse. Make it visible.

*Labels are a proxy of a proxy.* Training labels are assessor ratings of potential *dysgraphia* (a writing difficulty), which overlaps with, but is not the same as, dyslexia (a reading-based difficulty). Many people with dyslexia write neatly, and messy handwriting has many other causes (age, motor development, writing instrument, effort on the day).

*Inferred writer IDs.* If writer IDs are guessed from filenames, mistakes cause leakage (inflated metrics) or noise. If none exist, cross-validation is by image and all metrics are optimistic.

*Generalisation across languages and scripts.* Letter shapes, writing direction, the letters that are commonly reversed and schooling all differ between languages. A model trained on one language's sentences has no basis for judging another script.

*Children's data and consent.* Handwriting images of minors are personal data. Use them only within the dataset licence; don't publish raw images; obtain consent from parents/guardians (and assent from the child) before uploading new samples; keep processing in memory as the app does.

**4. How it maps to the project**

- `app/app.py`: the consent checkbox, the "not a diagnosis" warning, the privacy note and the "About this model and its limits" expander.
- `ckpt["trained_on"]`: honest provenance shown in the app.
- Phase 6 threshold, Phase 8 external test and Phase 9 heatmaps: the evidence base for every limitation you report.

**5. Common pitfalls**

- **Reporting a percentage as a probability of dyslexia.** It is a model score on dysgraphia-style labels, not a calibrated probability of anything clinical.
- **Ignoring subgroups.** Check whether errors concentrate in one age, school or image condition, where the data allows it.
- **Silent scope creep.** A tool built for "flag for a teacher to look at" gets used as "decide who gets help". Say clearly what it must not be used for.

*Deeper dive:* `theory/08_responsible_ai.md`.

## Phase 10: Streamlit app files

`app.py` and `requirements.txt` are maintained directly in `LexiAssist/app/`, next to `preprocess.py`, `model.py`, `explain.py` and the `lexiassist.pth` saved in Phase 7.

## Phase 11: Run and deploy

**Local test (recommended):** download the `LexiAssist/app/` folder from Drive, then in a terminal:

```bash
pip install -r requirements.txt
streamlit run app.py
```

**Test with real phone photos:** different lighting, angles and paper; blank pages; non-handwriting images. If results are fragile, add cropping or contrast normalisation to `preprocess.py` and retrain.

**Deploy (Streamlit Community Cloud or Hugging Face Spaces):**
1. Push the `app/` folder to a GitHub repo (use Git LFS if `lexiassist.pth` is large).
2. Create a new app on Streamlit Community Cloud pointing at `app.py`.
3. Keep the privacy and disclaimer text visible.

## Phase 12: Write-up checklist

- Datasets used, their sizes, how labels were created (assessor-labelled **potential dysgraphia**, not clinical dyslexia diagnoses), and the writer-ID limitation if any.
- Writer-aware 5-fold CV: results as mean ± std for SmallCNN vs ResNet18.
- Recall, specificity, F1, AUC, confusion matrix, calibration, and how the threshold was chosen.
- External-set results and what the drop (if any) says about generalisation.
- Grad-CAM / Integrated Gradients findings: strokes vs shortcuts.
- Ethics: screening not diagnosis, children's data, consent, no upload storage, possible bias (language, age, writing instrument).